# Intrinsic Wavefront Analysis and Plots

Interactive analysis of Double Zernike (DZ) focal-plane fits to FAM donut
wavefront data. Reads pre-computed fit results from `run_dz_fit.py`.

Supports multiple input file sets (chunks) that are concatenated for
combined analysis.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-02-23 | Aaron Roodman | Initial version |
| 2026-03-15 | Aaron Roodman | Added CCS/OCS coord_sys parameter; per-image linear fit; fit parameter plots; rotator angle filter |
| 2026-03-17 | Aaron Roodman | Switched to robust RLM; FAM block analysis; single-image residual maps; MPEG movie; multi-page PDFs |
| 2026-03-19 | Aaron Roodman | Generalized focal-plane Zernike fit; two fits per image: z1toz3 and z1toz6; merged output table; trio plots |
| 2026-03-19 | Aaron Roodman | Read Noll indices from visit_info; consistent iZs/iZidx indexing |
| 2026-03-20 | Aaron Roodman | Switched zk_intrinsic to coord_sys-specific column; added Z16+ plots |
| 2026-03-21 | Aaron Roodman | Fixed intrinsic Zernike units; added zk validation |
| 2026-03-23 | Aaron Roodman | Added prefix to parquet filename; added k1-k6 trio plots |
| 2026-03-24 | Aaron Roodman | Group by (el, rot) only; cap legend at 8 entries |
| 2026-03-25 | Aaron Roodman | Extracted fitting into code/dz_fitting.py module |
| 2026-04-06 | Aaron Roodman | Added make_per_day_pdfs and make_movie parameters; FAM block filtering; day dividers |
| 2026-04-10 | Aaron Roodman | Extracted fitting into standalone module code/dz_fitting.py |
| 2026-04-11 | Aaron Roodman | Major refactor: all plotting in code/dz_plotting.py; reads pre-computed fits; multi-input support; thermal correlations; DZ inter-correlation analysis; run_dz_plots.py CLI |
| 2026-04-12 | Aaron Roodman | Switched to HDF5 input (donuts+visits in single file); fit output as {stem}_fits.parquet |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Load Data](#load)
4. [Data Summary & Filtering](#summary)
5. [Reconstruct Fit Arrays](#reconstruct)
6. [Fit Parameter Plots](#fitplots)
7. [Single-Image Residual Maps](#singleimage)
8. [Trio Comparison Plots](#trio)
9. [Thermal Correlation Analysis](#thermal)
10. [DZ Inter-Correlation Analysis](#dzcorr)

<a id='params'></a>
## 1. Parameters

In [ ]:
from pathlib import Path

# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Coordinate system: 'CCS' or 'OCS' (must match mktable/fit choice)
coord_sys = 'OCS'

# === Multi-input discovery ===
# Glob pattern to find HDF5 files from intrinsics_mktable.
# Each HDF5 has donuts+visits tables; the matching fit file is
# {stem}_fits.parquet alongside it.
input_pattern = 'output/*_20260315_*.hdf5'

# Or specify explicit list of (hdf5_file, fit_file) tuples:
# input_pairs = [
#     ('output/run1.hdf5', 'output/run1_fits.parquet'),
# ]
input_pairs = None  # None = use input_pattern discovery

# === Plot flags (all True by default) ===
make_fit_param_plots = True
make_single_image_plots = True
make_trio_plots = True
make_thermal_analysis = True
make_dz_correlations = True
make_movie = True

# === Filtering parameters ===
rotator_angle_min = -90.0
rotator_angle_max = 90.0
min_seq_num_per_day = 5
position_group_tol = 3.0
bad_fit_threshold = 2.0  # μm
min_donuts = 200

# === Thermal analysis parameters ===
dz_terms_thermal = [
    (5, 5),   # focal astig45 of pupil astig45
    (6, 6),   # focal astig0 of pupil astig0
    (4, 4),   # focal defocus of pupil defocus
    (5, 10),  # focal astig45 of pupil trefoil
    (6, 9),   # focal astig0 of pupil trefoil
]
n_pca_components = 5

# === DZ correlation parameters ===
top_n_correlated = 10

# === Plotting parameters ===
plo_default = 4.0
phi_default = 96.0

# === Output ===
output_dir = 'output'
run_dir = None  # auto-derived from data dates

print('Parameters set.')

<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from astropy.table import QTable

from code.dz_fitting import derive_noll_indices, flag_bad_fits
from code.dz_plotting import (
    discover_input_pairs, load_and_concatenate, reconstruct_zk_fit,
    _identify_fam_blocks, _build_pointing_groups,
    plot_fit_params_and_residuals, plot_single_image_residual_grid,
    plot_zernike_trio,
    DEFAULT_THERMAL_VARS, dz_col_name,
    plot_thermal_scatter_grid, run_thermal_pca, plot_pca_loadings,
    run_pcr_analysis, plot_pcr_results, plot_thermal_importance,
    get_all_dz_columns, compute_dz_correlation_matrix,
    plot_dz_correlation_heatmap, get_top_correlated_pairs,
    plot_dz_scatter_top_pairs,
)

print('Imports complete.')

<a id='load'></a>
## 3. Load Data

Discovers and loads multiple HDF5 + fit parquet pairs, concatenating them
into combined tables for analysis. Each HDF5 contains donuts and visits
tables; the fit parquet contains DZ fit coefficients merged with visit info.

In [ ]:
# Discover or use explicit input pairs
if input_pairs is None:
    input_pairs = discover_input_pairs(input_pattern)
print(f'Discovered {len(input_pairs)} input pair(s):')
for hdf5_f, fit_f in input_pairs:
    print(f'  hdf5: {hdf5_f}')
    print(f'  fits: {fit_f}')

if not input_pairs:
    raise FileNotFoundError(
        f'No input pairs found for pattern: {input_pattern}')

# Load and concatenate
aosTable, fit_table, iZs, iZidx = load_and_concatenate(
    input_pairs, coord_sys)

# Derive plot Zernike subsets
iZs_plot_12 = iZs[:12] if len(iZs) >= 12 else iZs
iZs_plot_hi = iZs[12:]

# Auto-derive dates and output directory
all_day_obs = sorted(set(np.array(fit_table['day_obs']).tolist()))
date_range_str = (f'{all_day_obs[0]}-{all_day_obs[-1]}'
                  if len(all_day_obs) > 1 else str(all_day_obs[0]))

if run_dir is None:
    run_dir = f'{output_dir}/plots_{coord_sys}_{date_range_str}'
Path(run_dir).mkdir(parents=True, exist_ok=True)
print(f'\nOutput directory: {run_dir}')
print(f'Date range: {date_range_str}')
print(f'Zernike plot sets: Z4-Z15 ({len(iZs_plot_12)}), '
      f'Z16+ ({len(iZs_plot_hi)})')

<a id='summary'></a>
## 4. Data Summary & Filtering

In [ ]:
# Data summary
print(f'Donut table: {len(aosTable)} rows, {len(aosTable.columns)} columns')
print(f'Fit table: {len(fit_table)} visits, {len(fit_table.columns)} columns')
print(f'Day_obs values ({len(all_day_obs)}): {all_day_obs}')

# Count per day
fit_dobs = np.array(fit_table['day_obs'])
for dobs in all_day_obs:
    n = np.sum(fit_dobs == dobs)
    print(f'  {dobs}: {n} visits')

In [ ]:
# Filter donuts by rotator angle
if 'rotator_angle' in aosTable.colnames:
    rot_arr = np.array(aosTable['rotator_angle'])
    rot_mask = ((rot_arr >= rotator_angle_min)
                & (rot_arr <= rotator_angle_max))
    n_before = len(aosTable)
    aosTable = aosTable[rot_mask]
    print(f'Rotator angle filter [{rotator_angle_min}, {rotator_angle_max}]: '
          f'{n_before} -> {len(aosTable)} donuts')

# Filter sparse days
dobs_arr = np.array(aosTable['day_obs'])
snum_arr = np.array(aosTable['seq_num'])
keep_mask = np.ones(len(aosTable), dtype=bool)
for dobs in sorted(set(dobs_arr.tolist())):
    day_mask = dobs_arr == dobs
    n_unique = len(set(snum_arr[day_mask].tolist()))
    if n_unique < min_seq_num_per_day:
        keep_mask[day_mask] = False
        print(f'  Removed day {dobs}: only {n_unique} unique seq_num')
aosTable = aosTable[keep_mask]

# Filter to matched intra/extra donuts
if 'matched_intra_extra' in aosTable.colnames:
    matched_mask = np.array(aosTable['matched_intra_extra'])
    aosTable_matched = aosTable[matched_mask]
    print(f'Matched donuts: {len(aosTable_matched)} / {len(aosTable)}')
else:
    aosTable_matched = aosTable
    print(f'No matched_intra_extra column; using all {len(aosTable)} donuts')

# Bad-fit summary
fit_prefix = 'z1toz6'
bad_col = f'{fit_prefix}_bad_fit'
if bad_col in fit_table.colnames:
    n_bad = np.sum(np.array(fit_table[bad_col]))
    print(f'Bad fits ({bad_col}): {n_bad}/{len(fit_table)}')
elif 'bad_fit' in fit_table.colnames:
    bad_col = 'bad_fit'
    n_bad = np.sum(np.array(fit_table[bad_col]))
    print(f'Bad fits ({bad_col}): {n_bad}/{len(fit_table)}')

<a id='reconstruct'></a>
## 5. Reconstruct Fit Arrays

Rebuilds per-donut fit values from the stored per-image coefficients
by evaluating the focal-plane Zernike basis at each donut's field angle.

In [ ]:
reconstruct_zk_fit(aosTable_matched, fit_table, coord_sys, iZs,
                    prefix='z1toz3', max_focal_noll=3)
reconstruct_zk_fit(aosTable_matched, fit_table, coord_sys, iZs,
                    prefix='z1toz6', max_focal_noll=6)

# Alias for backward compatibility
aosTable_matched['zk_fit'] = aosTable_matched['zk_fit_z1toz3']

# Update day_obs arrays for matched table
day_obs_matched = np.array(aosTable_matched['day_obs'])

<a id='fitplots'></a>
## 6. Fit Parameter Plots

z1toz6 only. All pupil Zernike terms in output PDFs (Z4-Z15 first pages,
Z16+ second set). FAM block filtering applied to combined all-days plots.

In [ ]:
# Prepare good-fit subsets
if bad_col in fit_table.colnames:
    ft_good = fit_table[~np.array(fit_table[bad_col])]
else:
    ft_good = fit_table

# Build donut mask excluding bad-fit visits
bad_visits = set()
if bad_col in fit_table.colnames:
    for row in fit_table[np.array(fit_table[bad_col])]:
        bad_visits.add((int(row['day_obs']), int(row['seq_num'])))
if bad_visits:
    dobs_arr = np.array(aosTable_matched['day_obs'])
    snum_arr = np.array(aosTable_matched['seq_num'])
    good_donut_mask = np.array([
        (int(d), int(s)) not in bad_visits
        for d, s in zip(dobs_arr, snum_arr)])
else:
    good_donut_mask = np.ones(len(aosTable_matched), dtype=bool)

# Use fit_table as visit_info (it has visit metadata merged in)
visit_info = fit_table

# Pointing groups summary
_build_pointing_groups(ft_good, visit_info,
                       bin_size=position_group_tol, verbose=True)

In [ ]:
if make_fit_param_plots:
    for iZs_set, set_label in [(iZs_plot_12, 'Z4-Z15'),
                               (iZs_plot_hi, 'Z16+')]:
        if not iZs_set:
            continue
        print(f"\n{'='*60}")
        print(f'Fit parameter plots: {set_label} ({len(iZs_set)} Zernikes)')
        print(f"{'='*60}")

        # FAM block filter for combined plot
        block_mask = _identify_fam_blocks(ft_good, min_block_size=3)
        ft_blocks = ft_good[block_mask]
        n_excluded = len(ft_good) - len(ft_blocks)
        if n_excluded > 0:
            print(f'  Excluded {n_excluded} non-block FAM visits')

        block_visits = set(zip(
            np.array(ft_blocks['day_obs']).tolist(),
            np.array(ft_blocks['seq_num']).tolist()))
        block_donut_mask = good_donut_mask & np.array([
            (int(d), int(s)) in block_visits
            for d, s in zip(np.array(aosTable_matched['day_obs']),
                           np.array(aosTable_matched['seq_num']))])

        show_inline = (iZs_set is iZs_plot_12)
        plot_fit_params_and_residuals(
            ft_blocks, aosTable_matched, block_donut_mask,
            day_obs_list=all_day_obs, fit_prefix=fit_prefix,
            iZs_fit_plot=iZs_set, iZs_hist=iZs_set,
            iZs=iZs, iZidx=iZidx, coord_sys=coord_sys,
            visit_info=visit_info,
            position_group_tol=position_group_tol,
            output_dir=run_dir, show=show_inline)

<a id='singleimage'></a>
## 7. Single-Image Residual Maps

In [ ]:
if make_single_image_plots:
    # Build lookups from fit_table
    band_lookup = {}
    pointing_lookup = {}
    rot_col = ('rotAngle' if 'rotAngle' in fit_table.colnames
               else 'rotator_angle' if 'rotator_angle' in fit_table.colnames
               else None)
    if 'band' in fit_table.colnames:
        for row in fit_table:
            key = (int(row['day_obs']), int(row['seq_num']))
            band_lookup[key] = str(row['band'])
            if 'alt' in fit_table.colnames:
                ptg = {
                    'alt': float(row['alt']),
                    'az': float(row['az']),
                }
                if rot_col:
                    ptg['rotAngle'] = float(row[rot_col])
                pointing_lookup[key] = ptg

    all_images = sorted(set(zip(
        np.array(aosTable_matched['day_obs']).tolist(),
        np.array(aosTable_matched['seq_num']).tolist())))
    print(f'Generating residual maps for {len(all_images)} visits...')

    frame_files = []
    for i, (dobs, snum) in enumerate(all_images):
        band = band_lookup.get((dobs, snum), '')
        ptg = pointing_lookup.get((dobs, snum), {})
        show_inline = (i == 0)
        outfile = plot_single_image_residual_grid(
            aosTable_matched, dobs, snum,
            iZs=iZs, iZidx=iZidx, coord_sys=coord_sys,
            band=band,
            alt=ptg.get('alt'), az=ptg.get('az'),
            rotAngle=ptg.get('rotAngle'),
            fit_table=fit_table, fit_prefix='z1toz3',
            iZs_plot=iZs_plot_12,
            output_dir=run_dir, show=show_inline)
        if outfile is not None:
            frame_files.append(outfile)
        if (i + 1) % 50 == 0:
            print(f'  ...processed {i + 1}/{len(all_images)} visits')

    print(f'Generated {len(frame_files)} residual map frames')

    # Movie creation
    movie_success = False
    if len(frame_files) > 1:
        list_file = f'{run_dir}/frame_list.txt'
        with open(list_file, 'w') as f:
            for fpath in frame_files:
                f.write(f"file '{Path(fpath).name}'\n")
                f.write('duration 0.5\n')

        ffmpeg_cmd = ['ffmpeg', '-y', '-f', 'concat', '-safe', '0',
                      '-i', 'frame_list.txt',
                      '-vf', 'pad=ceil(iw/2)*2:ceil(ih/2)*2',
                      '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
                      '-r', '2', 'single_image_residuals.mp4']

        if make_movie:
            try:
                result = subprocess.run(
                    ffmpeg_cmd, capture_output=True, text=True,
                    cwd=run_dir)
                if result.returncode == 0:
                    print(f'Saved movie: {run_dir}/single_image_residuals.mp4')
                    movie_success = True
                else:
                    print(f'ffmpeg failed: {result.stderr[-300:]}')
            except FileNotFoundError:
                print('ffmpeg not found')
        else:
            print(f'Movie creation skipped. To create manually:')
            print(f"  cd {run_dir} && {' '.join(ffmpeg_cmd)}")

        if movie_success:
            for fpath in frame_files:
                Path(fpath).unlink(missing_ok=True)
            Path(list_file).unlink(missing_ok=True)
            print(f'Cleaned up {len(frame_files)} JPEGs')
        else:
            print('Keeping individual JPEGs.')

<a id='trio'></a>
## 8. Trio Comparison Plots

Data / Model / Data-Model maps for each pupil Zernike, with both
z1toz3 and z1toz6 fit subtraction variants.

In [ ]:
if make_trio_plots:
    # Filter bad-fit donuts
    if bad_visits:
        dobs_arr = np.array(aosTable_matched['day_obs'])
        snum_arr = np.array(aosTable_matched['seq_num'])
        good_mask = np.array([
            (int(d), int(s)) not in bad_visits
            for d, s in zip(dobs_arr, snum_arr)])
        aos_good = aosTable_matched[good_mask]
    else:
        aos_good = aosTable_matched

    # z1toz3 trio
    trio_pdf = f'{run_dir}/trio_comparison_all.pdf'
    with PdfPages(trio_pdf) as pdf:
        for i, iZ in enumerate(iZs):
            plot_zernike_trio(
                aos_good, iZ, iZs=iZs, iZidx=iZidx,
                coord_sys=coord_sys, fit_prefix='z1toz3',
                date_range_str=date_range_str,
                pdf=pdf, show=(i == 0))
    print(f'Saved: {trio_pdf}')

    # z1toz6 trio
    trio_k6_pdf = f'{run_dir}/trio_comparison_k1to6_all.pdf'
    with PdfPages(trio_k6_pdf) as pdf:
        for iZ in iZs:
            plot_zernike_trio(
                aos_good, iZ, iZs=iZs, iZidx=iZidx,
                coord_sys=coord_sys, fit_prefix='z1toz6',
                date_range_str=date_range_str,
                pdf=pdf, show=False)
    print(f'Saved: {trio_k6_pdf}')

<a id='thermal'></a>
## 9. Thermal Correlation Analysis

Scatter plots of selected DZ coefficients vs thermal variables, followed
by PCA-based regression to identify which thermal modes drive each DZ term.

In [ ]:
if make_thermal_analysis:
    fit_df = fit_table.to_pandas()
    thermal_vars = [tv for tv in DEFAULT_THERMAL_VARS
                    if tv in fit_df.columns]

    if not thermal_vars:
        print('No thermal variables found in fit table — skipping.')
        make_thermal_analysis = False
    else:
        print(f'Thermal variables ({len(thermal_vars)}): {thermal_vars}')

        # Quality cuts for thermal analysis
        if bad_col in fit_df.columns:
            fit_df = fit_df[~fit_df[bad_col]].copy()
            print(f'After bad-fit cut: {len(fit_df)} visits')

        thermal_pdf = f'{run_dir}/thermal_correlations.pdf'
        with PdfPages(thermal_pdf) as pdf:
            plot_thermal_scatter_grid(
                fit_df, dz_terms_thermal, thermal_vars,
                dz_prefix='z1toz6', pdf=pdf, show=True)
        print(f'Saved: {thermal_pdf}')

In [ ]:
if make_thermal_analysis:
    scaler, pca, pc_scores, valid_mask = run_thermal_pca(
        fit_df, thermal_vars, n_pca_components)

    pcr_results = run_pcr_analysis(
        fit_df, dz_terms_thermal, pca, pc_scores, valid_mask)

    pca_pdf = f'{run_dir}/thermal_pca_regression.pdf'
    with PdfPages(pca_pdf) as pdf:
        plot_pca_loadings(pca, thermal_vars, pdf=pdf, show=True)
        plot_pcr_results(pcr_results, pdf=pdf, show=True)
        plot_thermal_importance(pcr_results, pca, thermal_vars,
                               pdf=pdf, show=True)
    print(f'Saved: {pca_pdf}')

<a id='dzcorr'></a>
## 10. DZ Inter-Correlation Analysis

Full correlation matrix heatmap across all DZ coefficients (k,j), plus
scatter plots of the top N most correlated pairs. This reveals physical
couplings between focal-plane aberration modes, e.g. (5,5) vs (6,6)
(the two astigmatism modes).

In [ ]:
if make_dz_correlations:
    # Use fit_df from thermal section, or create fresh
    if 'fit_df' not in dir() or fit_df is None:
        fit_df = fit_table.to_pandas()
        if bad_col in fit_df.columns:
            fit_df = fit_df[~fit_df[bad_col]].copy()

    dz_cols, dz_labels = get_all_dz_columns(fit_table, prefix='z1toz6')
    print(f'DZ coefficients: {len(dz_cols)} columns')

    corr_matrix = compute_dz_correlation_matrix(fit_df, dz_cols)

    corr_pdf = f'{run_dir}/dz_correlations.pdf'
    with PdfPages(corr_pdf) as pdf:
        plot_dz_correlation_heatmap(corr_matrix, dz_labels,
                                   pdf=pdf, show=True)

        top_pairs = get_top_correlated_pairs(
            corr_matrix, dz_cols, dz_labels,
            top_n=top_n_correlated)

        print(f'\nTop {len(top_pairs)} correlated DZ pairs:')
        for col_i, col_j, lab_i, lab_j, r in top_pairs:
            print(f'  {lab_i} vs {lab_j}: r = {r:.3f}')

        plot_dz_scatter_top_pairs(fit_df, top_pairs,
                                 pdf=pdf, show=True)

    print(f'Saved: {corr_pdf}')